AI Resume Screening System with Tracing

In [ ]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Resume Screening System"

In [ ]:
!pip install -q transformers==4.41.2 accelerate sentencepiece langchain langchain-core langchain-community

In [ ]:
# 📦 1. IMPORTS
# =========================
from transformers import pipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import HuggingFacePipeline

In [ ]:
# 🤖 2. LOAD MODEL (STABLE)
# =========================
hf_pipeline = pipeline(
    task="text2text-generation",
    model="google/flan-t5-small",   # lightweight, no crash
    device=-1
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)
parser = StrOutputParser()


In [ ]:
# 📄 3. INPUT DATA (3 RESUMES + JD)
# =========================
job_desc = """
Data Scientist role requiring:
Python, Machine Learning, SQL, Data Visualization, 2+ years experience
"""

resume_strong = """
3 years experience in Data Science.
Skills: Python, Machine Learning, SQL, Data Visualization.
Worked on multiple ML projects.
"""

resume_avg = """
1 year experience.
Skills: Python, basic Machine Learning.
"""

resume_weak = """
Fresher.
Skills: Communication, MS Word.
"""


In [ ]:
# 🧠 4. PROMPTS
# =========================
extract_prompt = PromptTemplate.from_template("""
Extract skills, experience, and tools from resume.

Resume:
{resume}

Answer:
skills:
experience:
tools:
""")

match_prompt = PromptTemplate.from_template("""
Compare resume with job description.

Resume:
{resume_data}

Job Description:
{job_desc}

Answer:
matched_skills:
missing_skills:
""")

score_prompt = PromptTemplate.from_template("""
Give a score from 0 to 100 based on matching.

Match Data:
{match_data}

Answer:
score:
""")

explain_prompt = PromptTemplate.from_template("""
Explain the score in 2 lines.

Score:
{score}

Match Data:
{match_data}
""")

In [ ]:
# 🔗 5. CHAINS (LCEL)
# =========================
extract_chain = extract_prompt | llm | parser
match_chain = match_prompt | llm | parser
score_chain = score_prompt | llm | parser
explain_chain = explain_prompt | llm | parser

In [ ]:
def run_pipeline(resume, job_desc):

    extracted = extract_chain.invoke({"resume": resume})

    resume_text = resume.lower()

    required_skills = ["python", "machine learning", "sql", "data visualization"]

    matched_skills = [skill for skill in required_skills if skill in resume_text]
    missing_skills = [skill for skill in required_skills if skill not in resume_text]

    matched = f"Matched Skills: {matched_skills}\nMissing Skills: {missing_skills}"

    score_value = int((len(matched_skills) / len(required_skills)) * 100)
    score = str(score_value)

    # ✅ explanation
    explanation = f"""
The candidate scored {score} because they have {len(matched_skills)} out of {len(required_skills)} required skills.
Matched skills: {matched_skills}.
Missing skills: {missing_skills}.
"""

    return {
        "Extracted": extracted.strip(),
        "Matched": matched,
        "Score": score,
        "Explanation": explanation.strip()
    }

In [ ]:
print("STRONG CANDIDATE\n", run_pipeline(resume_strong, job_desc))
print("\n" + "="*60)

print("AVERAGE CANDIDATE\n", run_pipeline(resume_avg, job_desc))
print("\n" + "="*60)

print("WEAK CANDIDATE\n", run_pipeline(resume_weak, job_desc))

STRONG CANDIDATE
 {'Extracted': 'a) Python, Machine Learning, SQL, Data Visualization.', 'Matched': "Matched Skills: ['python', 'machine learning', 'sql', 'data visualization']\nMissing Skills: []", 'Score': '100', 'Explanation': "The candidate scored 100 because they have 4 out of 4 required skills.\nMatched skills: ['python', 'machine learning', 'sql', 'data visualization'].\nMissing skills: []."}

AVERAGE CANDIDATE
 {'Extracted': 'a) Python, basic Machine Learning.', 'Matched': "Matched Skills: ['python', 'machine learning']\nMissing Skills: ['sql', 'data visualization']", 'Score': '50', 'Explanation': "The candidate scored 50 because they have 2 out of 4 required skills.\nMatched skills: ['python', 'machine learning'].\nMissing skills: ['sql', 'data visualization']."}

WEAK CANDIDATE
 {'Extracted': 'a computer', 'Matched': "Matched Skills: []\nMissing Skills: ['python', 'machine learning', 'sql', 'data visualization']", 'Score': '0', 'Explanation': "The candidate scored 0 because t

Explanation of Approach

In this project, an AI-powered resume screening system was developed using LangChain. The system follows a structured pipeline consisting of skill extraction, matching, scoring, and explanation.

For skill extraction, a Large Language Model (LLM) was used to identify relevant information such as skills, experience, and tools from the resume. To ensure accuracy and avoid hallucination, clear and structured prompts were designed.

Instead of relying entirely on the LLM for scoring, a rule-based approach was implemented for matching and scoring. This was done because LLM-based scoring was found to be inconsistent. The rule-based method compares extracted skills with job requirements and assigns a score based on the percentage of matched skills, ensuring reliable and interpretable results.

The system also provides an explanation for each score, making the output transparent and easy to understand. Additionally, prompt refinement was performed to debug incorrect outputs and improve extraction quality.

Overall, the system successfully demonstrates how LLMs and traditional logic can be combined to build a robust and explainable AI solution.